### Create work-funder associations from Crossref metadata

Extracts the `funders` array from Crossref works in `locations_mapped`, matches funder DOIs to `openalex.common.funder`, and writes a junction table analogous to `openalex.mid.work_funder`.

This table feeds into:
- **CreateWorksEnriched** (4th funder union leg)
- **WorkAwards** (bridge crossref award_ids to `openalex_awards`)

### Create missing funders

Crossref funder DOIs not yet in `openalex.common.funder` get new funder records
in `openalex.mid.funder`. IDs are minted sequentially from `MAX(funder_id) + 1`.
For each DOI, we pick the most common display name across works.

In [ ]:
%sql
-- Preview: funders to be created (top 30 by work count)
WITH crossref_funder_dois AS (
  SELECT
    f.doi AS funder_doi,
    f.name AS funder_name,
    COUNT(DISTINCT lm.work_id) AS work_count
  FROM openalex.works.locations_mapped lm
  LATERAL VIEW EXPLODE(funders) AS f
  WHERE lm.provenance = 'crossref'
    AND lm.work_id IS NOT NULL
    AND f.doi IS NOT NULL
    AND f.doi NOT LIKE '%#%'
    AND f.doi NOT LIKE '%https://%'
    AND f.name IS NOT NULL AND f.name != '' AND f.name != '#'
  GROUP BY f.doi, f.name
),
-- Pick the most common name per DOI
best_name AS (
  SELECT funder_doi, funder_name, work_count,
    ROW_NUMBER() OVER (PARTITION BY funder_doi ORDER BY work_count DESC) AS rn
  FROM crossref_funder_dois
),
unmatched AS (
  SELECT bn.funder_doi, bn.funder_name, bn.work_count
  FROM best_name bn
  LEFT JOIN openalex.mid.funder cf ON bn.funder_doi = cf.doi
  WHERE bn.rn = 1 AND cf.funder_id IS NULL
)
SELECT * FROM unmatched ORDER BY work_count DESC LIMIT 30

In [ ]:
%sql
-- Create new funder records in openalex.mid.funder
WITH crossref_funder_dois AS (
  SELECT
    f.doi AS funder_doi,
    f.name AS funder_name,
    COUNT(DISTINCT lm.work_id) AS work_count
  FROM openalex.works.locations_mapped lm
  LATERAL VIEW EXPLODE(funders) AS f
  WHERE lm.provenance = 'crossref'
    AND lm.work_id IS NOT NULL
    AND f.doi IS NOT NULL
    AND f.doi NOT LIKE '%#%'
    AND f.doi NOT LIKE '%https://%'
    AND f.name IS NOT NULL AND f.name != '' AND f.name != '#'
  GROUP BY f.doi, f.name
),
best_name AS (
  SELECT funder_doi, funder_name, work_count,
    ROW_NUMBER() OVER (PARTITION BY funder_doi ORDER BY work_count DESC) AS rn
  FROM crossref_funder_dois
),
unmatched AS (
  SELECT bn.funder_doi, bn.funder_name
  FROM best_name bn
  LEFT JOIN openalex.mid.funder cf ON bn.funder_doi = cf.doi
  WHERE bn.rn = 1 AND cf.funder_id IS NULL
),
max_id AS (
  SELECT MAX(funder_id) AS current_max FROM openalex.mid.funder
),
minted AS (
  SELECT
    (SELECT current_max FROM max_id) + ROW_NUMBER() OVER (ORDER BY funder_doi) AS funder_id,
    funder_name AS display_name,
    '[]' AS alternate_titles,
    CAST(NULL AS STRING) AS country_code,
    CAST(NULL AS STRING) AS description,
    CAST(NULL AS STRING) AS homepage_url,
    CAST(NULL AS STRING) AS image_url,
    CAST(NULL AS STRING) AS image_thumbnail_url,
    CAST(NULL AS STRING) AS ror_id,
    CAST(NULL AS STRING) AS wikidata_id,
    TRY_CAST(REPLACE(funder_doi, '10.13039/', '') AS BIGINT) AS crossref_id,
    funder_doi AS doi,
    CURRENT_TIMESTAMP() AS created_date
  FROM unmatched
)
INSERT INTO openalex.mid.funder
SELECT * FROM minted

In [ ]:
%sql
-- Sanity check: no duplicate funder_ids in mid.funder
SELECT COUNT(*) AS duplicate_funder_ids
FROM (
  SELECT funder_id, COUNT(*) AS n
  FROM openalex.mid.funder
  GROUP BY funder_id
  HAVING n > 1
)

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.crossref_work_funders
USING delta
AS
WITH exploded AS (
  SELECT
    lm.work_id,
    f.doi AS funder_doi,
    f.name AS funder_name,
    f.awards AS award_ids
  FROM openalex.works.locations_mapped lm
  LATERAL VIEW EXPLODE(funders) AS f
  WHERE lm.provenance = 'crossref'
    AND lm.work_id IS NOT NULL
    AND f.doi IS NOT NULL
),
matched AS (
  SELECT
    e.work_id,
    cf.funder_id,
    e.award_ids
  FROM exploded e
  JOIN openalex.mid.funder cf ON e.funder_doi = cf.doi
)
-- Deduplicate: one row per (work_id, funder_id), merge award_ids
SELECT
  work_id,
  funder_id,
  ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(COALESCE(award_ids, ARRAY())))) AS award_ids
FROM matched
GROUP BY work_id, funder_id

### Sanity checks

In [ ]:
%sql
-- Row counts and uniqueness
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT CONCAT(work_id, ':', funder_id)) AS distinct_keys,
  COUNT(DISTINCT work_id) AS distinct_works,
  COUNT(DISTINCT funder_id) AS distinct_funders,
  COUNT(CASE WHEN SIZE(award_ids) > 0 THEN 1 END) AS rows_with_awards
FROM openalex.awards.crossref_work_funders

In [ ]:
%sql
-- Verify composite key uniqueness (should return 0)
SELECT COUNT(*) AS duplicate_keys
FROM (
  SELECT work_id, funder_id, COUNT(*) AS n
  FROM openalex.awards.crossref_work_funders
  GROUP BY work_id, funder_id
  HAVING n > 1
)

### Insert awards into `openalex_awards_raw`

Follows the same pattern as `CreateBackfillAwards`: explode award_ids, generate xxhash64-based IDs, insert with provenance and priority.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW crossref_work_funder_awards AS
WITH exploded AS (
  SELECT DISTINCT
    cwf.funder_id,
    LOWER(award_id) AS normalized_award_id,
    award_id AS funder_award_id
  FROM openalex.awards.crossref_work_funders cwf
  LATERAL VIEW EXPLODE(award_ids) AS award_id
  WHERE SIZE(award_ids) > 0
),
funders AS (
  SELECT funder_id, display_name, ror_id, doi
  FROM openalex.mid.funder
)
SELECT
  ABS(XXHASH64(CONCAT(f.funder_id, ':', e.normalized_award_id))) % 9000000000 AS id,
  CAST(NULL AS STRING) AS display_name,
  CAST(NULL AS STRING) AS description,
  f.funder_id,
  e.funder_award_id,
  CAST(NULL AS DOUBLE) AS amount,
  CAST(NULL AS STRING) AS currency,
  STRUCT(
    CONCAT('https://openalex.org/F', f.funder_id) AS id,
    f.display_name,
    f.ror_id,
    f.doi
  ) AS funder,
  CAST(NULL AS STRING) AS funding_type,
  CAST(NULL AS STRING) AS funder_scheme,
  'crossref_work_funders' AS provenance,
  CAST(NULL AS DATE) AS start_date,
  CAST(NULL AS DATE) AS end_date,
  CAST(NULL AS INT) AS start_year,
  CAST(NULL AS INT) AS end_year,
  CAST(NULL AS STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >) AS lead_investigator,
  CAST(NULL AS STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >) AS co_lead_investigator,
  CAST(NULL AS ARRAY<STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >>) AS investigators,
  CAST(NULL AS STRING) AS landing_page_url,
  CAST(NULL AS STRING) AS doi,
  CONCAT('https://api.openalex.org/works?filter=awards.id:G', ABS(XXHASH64(CONCAT(f.funder_id, ':', e.normalized_award_id))) % 9000000000) AS works_api_url,
  CURRENT_TIMESTAMP() AS created_date,
  CURRENT_TIMESTAMP() AS updated_date
FROM exploded e
JOIN funders f ON f.funder_id = e.funder_id

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'crossref_work_funders' AND priority = 2

In [ ]:
%sql
INSERT INTO openalex.awards.openalex_awards_raw
SELECT *, 2 AS priority
FROM crossref_work_funder_awards